# Simple ReAct Agent using LangChain and OpenRouter
#### Authoerd by Dr.Tiziana Ligorio for *AI Agents - CSCI 395.32* taught at Hunter College of The City University of New York

#### This demo has two parts.  

#### In **Part 1**
We build a simple ReAct agent using the LangChain framework to illustrate how much a framework can abstract and automate compared to our previous demo, where we implemented the agent loop from scratch.

Note that, because a lot of the agent logic is built within the framework, as a designer, you have much less control over context engineering. We will discuss context engineering in depth in future lectures.

As before, the agent follows the ReAct pattern by iteratively alternating between **reasoning** and **acting** to accomplish a task. However, in this version most of the control flow, such as managing the reasoning loop, invoking tools, and handling intermediate state, is handled by the framework (and thus standardized) rather than implemented directly by the agent developer.  
Here are the [LangChain docs](https://docs.langchain.com/oss/python/langchain/overview) for your reference.

For this tutorial, we will demonstrate access to search tools (web search and Wikipedia), which it can use to gather information and answer user queries. We will then show how to include a tool for fetching via MCP, and a custom tool for filtering.  

#### In **Part 2** 
We demonstrate how to trace and run evals using LangSmith. This part will be covered after our lecture on evals.  
Here are the [LangSmith docs](https://docs.langchain.com/langsmith/home) for your reference.

<img src="https://raw.githubusercontent.com/tligorio/langchain_react_agent_tutorial/main/images/lang_ReAct_agent.png" alt="ReAct Agent with LanghChain" width="60%"/>


# Installs and Imports

In [1]:
%%capture
!pip install langchain langchain-openai langchain-community langsmith google-search-results wikipedia


`%%capture` hides the output

In [2]:
# LangChain: the core framework
# Defines agents, executors, prompts, and how LLMs + tools are orchestrated
from langchain.agents import create_agent
from langchain_core.tools import tool

# LangChain Community:
# Contains third-party tools like Wikipedia, search APIs, web loaders, etc.
from langchain_community.agent_toolkits.load_tools import load_tools

# LangSmith: observability + prompt hub
# Used here to fetch a pre-built ReAct prompt (and possibly for tracing/debugging)
from langsmith import Client

# OpenAI integration for LangChain
# Provides ChatOpenAI, a LangChain-compatible LLM wrapper
from langchain_openai import ChatOpenAI

# Standard library
import os
import requests
import re

**OpenRouter** is a unified API that provides access to various LLMs through a single interface. It offers a generous free tier and affordable token usage for minimal cost, making it ideal for learning and experimentation.

If you already pay for other LLM providers or prefer to use a different service, you are welcome to adapt the code accordingly.

# Setup your API Keys

## Step 1 — Get an OpenRouter API key
For this demo we will use an LLM via OpenRouter, which requires an API key.

1. Go to https://openrouter.ai

2. Sign in (or create an account if you don't have one)

3. Once logged in, navigate to https://openrouter.ai/settings/keys

4. Click Create Key

5. Give the key a name, e.g. colab-langchain_ReAct_agent

6. Copy the key immediately (you won't be able to see it again)

**Important:
Treat this key like a password.
Do not share it, paste it into notebooks, or commit it to GitHub.**


## Step 2 — Add a secret in Colab (UI)

1.  On the left sidebar, click 🔑 Secrets

2.  Add a new secret:


*   Name: OPENROUTER_API_KEY
*   Value: your actual API key

3. Toggle the switch to the left to give notebook access (you should see a checkmark)

## If running locally — Add a secret in .env

1. Create a .env file in the project root:

`touch .env`

2.  Add the following (replace with your own key):

`OPENROUTER_API_KEY=your_openrouter_key_here`. 


**Important: Never paste API keys into code cells.**

## Step 3 — Get a SerpAPI key

SerpAPI is used to access search tools (e.g. Google search) from agents.
It requires a separate account and API key.

1. Go to https://serpapi.com

2. Click Sign up

3. Create an account or login if you have one (SerpAPI offers a free tier that is sufficient for this course.)

4. Confirm your email and phone number if prompted

5. Once logged in, go to your dashboard (https://serpapi.com/dashboard)

6. Locate and copy your API Key

## Step 4 — Add a secret in Colab (UI)

1.  On the left sidebar, click 🔑 Secrets

2.  Add a new secret:


*   Name: SERPAPI_API_KEY
*   Value: your actual API key

3. Toggle the switch to the left to give notebook access (you should see a checkmark)

## If running locally — Add a secret in .env

1.  Add the following to .env (replace with your own key):

`SERPAPI_API_KEY=your_serpapi_key_here`. 

**Important: Never paste API keys into code cells.**

## Load the API Keys 

### In Colab (For Part 1 Only):
Uncomment and run the cell below if you're using Google Colab.

In [3]:
# # Load API keys from Colab Secrets into environment variables if running in colab
# from google.colab import userdata

# keys = ["OPENROUTER_API_KEY", "SERPAPI_API_KEY"]

# for key in keys:
#     value = userdata.get(key)
#     assert value is not None, f"{key} not found in Colab Secrets or access is disabled"
#     os.environ[key] = value

# print("API keys successfully loaded")


### Locally:

In [4]:
# Load API keys from .env if running locally - see local install instructions in the repo README
from dotenv import load_dotenv
load_dotenv()


True

In [5]:
# sanity check
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))
print("SERPAPI_API_KEY present:", bool(os.getenv("SERPAPI_API_KEY")))

OPENROUTER_API_KEY present: True
SERPAPI_API_KEY present: True


### These will be used in Part 2. If you are running Part 1 only, comment these out.

In [6]:
# Configure LangSmith tracing here, before any LangChain objects are created.
# Setting these after the kernel has already run agent calls can cause
# a "ValueError: I/O operation on closed file" due to async thread conflicts
# between LangSmith's tracing backend and Jupyter's output stream.
if not os.getenv("LANGSMITH_API_KEY"):
    print("Warning: LANGSMITH_API_KEY not set; tracing disabled")
else:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_PROJECT"] = "react-agent-demo"
    os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

# Part 1: ReAct Agent with LangChain

## Load the tools our Agent will have access to

In [7]:
 # Load LangChain's built-in wrappers for SerpAPI (web search) and Wikipedia as agent tools  
tools = load_tools(["serpapi", "wikipedia"])

## Load the LLM and setup the Agent

In this demo we use the gpt-4o-mini model via OpenRouter. This is a lightweight and cost-efficient model that works well for agentic workflows involving tool use. You may change the model by selecting one available on the [OpenRouter model list](https://openrouter.ai/models). Setting the temperature to zero makes the agent's behavior deterministic, which is we choose for demos and debugging: given the same input, the agent will tend to make the same decisions and tool calls.

In [8]:
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    max_tokens=600,
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY")
)

In [9]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a research assistant.\n"
        "For research questions, do not rely on a single source.\n"
        "Use multiple search or information-gathering tool calls to cover "
        "different time periods, perspectives, or subtopics before answering.\n"
        "Only provide a final answer after you have gathered information "
        "from multiple sources."
        )
    )

That is all we need to do to setup the Agent! LangChain will handle the logic. Note how short our system prompt is now. The short system prompt is enough because the framework now implements the ReAct loop.

In our previous demo (ReAct from scratch), the LLM drove the ReAct loop and the system prompt had to explicity specify:
| Aspect | Responsibilities |
|------|------------------|
| **Control loop** | Thought → Action → Observation<br>Repeat up to `max_iterations` |
| **Action protocol** | JSON format<br>One action at a time |
| **Tool routing** | Tool names<br>Tool schemas |
| **Termination** | When to stop<br>Exact Final Answer format |
| **Error prevention** | Rounding rules<br>Allowed operations<br>Strict formatting constraints |
| **Prompt role** | Program<br>State machine<br>Protocol definition |




 With a framework instead:
* The logic is no longer in the prompt
* The logic `LLM → decide → tool → observe → repeat → stop` is implemented in code
* The framwork handles:
    * Iteration (including how many times it can loop)

    * Tool invocation (what tools exist and how to call them)

    * Passing observations back to the model

    * Termination conditions

    * Output structure

When we use an agent framework, the reasoning loop moves from the prompt into code, so the prompt can be short and declarative.  

As a developer, it is however important to understand exactly how it works before relying on the abstractions afforded by the frameworks.

## Run the Agent

In [10]:
answer = agent.invoke({
    "messages": [
        {"role": "user", "content": "Who is Tiziana Ligorio?"}
    ]
})
answer

{'messages': [HumanMessage(content='Who is Tiziana Ligorio?', additional_kwargs={}, response_metadata={}, id='f0bbc334-edb7-40d8-ad60-3f75363164db'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 188, 'total_tokens': 241, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 6e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 6e-05, 'upstream_inference_prompt_cost': 2.82e-05, 'upstream_inference_completions_cost': 3.18e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'gen-1772016526-2fYbbAxIOIEbiFHfqDto', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c946a-6580-7de3-abd4-cca074992db3-0', tool_calls=[{'name': 'wikipedia', 'args': {'q

This is too much to look at! Let's break it down slowly.  

When we call agent.invoke, the returned object contains the full internal execution state of the agent, including:
* messages
* tool calls
* tool outputs
* metadata
* the final response.   

We use `build_agent_trace(result)` to reorganize this raw output into a readable, step-by-step trace that makes it easier to inspect what the agent did, which tools it called, and how it arrived at its final answer. 

Let's inspect the components of the answer first to understand how to extract data from it to build the trace.

The answer has a list of messages, and each message has a type (the node that issued it)

In [11]:
for msg in answer["messages"]:
    print(msg.type)

human
ai
tool
tool
ai


AI messages may issue a tool call

In [12]:
for msg in answer["messages"]:
    mtype = msg.type
    if mtype == "ai":
            # Action(s): tool calls
            if getattr(msg, "tool_calls", None):
                for call in msg.tool_calls:
                    print(call)
                    # trace.append({"type": "action", "tool": call["name"], "args": call["args"]})

{'name': 'wikipedia', 'args': {'query': 'Tiziana Ligorio'}, 'id': 'call_6x6HflVBwMrw25rhkE9qxHy1', 'type': 'tool_call'}
{'name': 'Search', 'args': {'__arg1': 'Tiziana Ligorio news'}, 'id': 'call_FA7VthLMUz35MNYGVWM1ktCI', 'type': 'tool_call'}


Tool messages contain observations: the data returned by the tool. Rather than storing the full tool output (msg.content) in the trace, which can be large and clutter the display, we record only the length of the result. This keeps the trace compact and easier to inspect while still conveying how much information the tool returned.

In [13]:
for msg in answer["messages"]:
    mtype = msg.type
    if mtype == "tool":
        print(msg)
        # trace.append({"type": "observation", "tool": msg.name, "chars": n})

content="Page: Warren Neidich\nSummary: Warren Neidich ( NYE-dik) is an American artist who lives in Berlin and Los Angeles. He was a professor at Kunsthochschule Weißensee School of Art, Berlin and visiting scholar at Otis College of Art and Design, Los Angeles.\nNeidich is founding director of the Saas-Fee Summer Institute of Art (SFSIA). He has collaborated with artists, curators and critics including: Barry Schwabsky (co-director of SFSIA), Armen Avanessian, Nicolas Bourriaud, Tiziana Terranova, Franco Berardi, Hans-Ulrich Obrist, Isaac Julien, Hito Steyerl, Chris Kraus (American writer), and many others.\nHis work has been exhibited at numerous institutions including: MoMA PS1, Whitney Museum of American Art, LACMA – Los Angeles County Museum of Art, California Museum of Photography, ICA – Institute of Contemporary Arts, London, Museum Ludwig, Cologne, and Walker Art Center, Minneapolis, Minnesota.\nIn relation to his exhibitions and extended theories he has edited and published o

Now that we have broken down the structure of the answer, let's parse it into a step-by-step trace that makes it easier to inspect the agent's actions.

In [14]:
def build_agent_trace(result):
    trace = []
    for msg in result["messages"]:
        if msg.type == "human":
            trace.append({"type": "user", "content": msg.content})

        elif msg.type == "ai" and getattr(msg, "tool_calls", None):
            for call in msg.tool_calls:
                trace.append({"type": "action", "tool": call["name"], "args": call["args"]})

        elif msg.type == "tool":
            n = len(msg.content) if isinstance(msg.content, str) else None
            trace.append({"type": "observation", "tool": msg.name, "chars": n})

    final = result["messages"][-1].content
    return {"trace": trace, "final_answer": final}


In [15]:
trace = build_agent_trace(answer)
trace

{'trace': [{'type': 'user', 'content': 'Who is Tiziana Ligorio?'},
  {'type': 'action',
   'tool': 'wikipedia',
   'args': {'query': 'Tiziana Ligorio'}},
  {'type': 'action',
   'tool': 'Search',
   'args': {'__arg1': 'Tiziana Ligorio news'}},
  {'type': 'observation', 'tool': 'wikipedia', 'chars': 1521},
  {'type': 'observation', 'tool': 'Search', 'chars': 1340}],
 'final_answer': 'Tiziana Ligorio is a Doctoral Lecturer in the Department of Computer Science at Hunter College, part of The City University of New York. She has a PhD from The Graduate Center of the City University of New York and specializes in areas such as machine learning and spoken language processing. \n\nIn addition to her teaching role, she is involved in research and has contributed to discussions on the implications of large language models (LLMs) in computing. Her work is recognized in the academic community, particularly for her contributions to machine learning education.\n\nThere is limited information availa

Much easier to trace now!   

From this trace alone, it’s not obvious that the Wikipedia lookup was irrelevant (there is no Wikipedia page for me), which is why deeper inspection is sometimes necessary when results look questionable. Recognizing and investigating such mismatches is an important skill when debugging agent behavior.  

Now let's try to get our agent to dig a little deeper.

In [16]:
inputs = {"messages": [{"role": "user", "content": "Who is Tiziana Ligorio? Dig out different areas she has worked on from her early PhD years until now."}]}

answer = agent.invoke(inputs)
trace = build_agent_trace(answer)
trace

{'trace': [{'type': 'user',
   'content': 'Who is Tiziana Ligorio? Dig out different areas she has worked on from her early PhD years until now.'},
  {'type': 'action',
   'tool': 'wikipedia',
   'args': {'query': 'Tiziana Ligorio'}},
  {'type': 'action',
   'tool': 'Search',
   'args': {'__arg1': 'Tiziana Ligorio research areas and contributions'}},
  {'type': 'observation', 'tool': 'wikipedia', 'chars': 1521},
  {'type': 'observation', 'tool': 'Search', 'chars': 1532}],
 'final_answer': "Tiziana Ligorio is a researcher and educator in the field of computer science, particularly known for her work on machine learning and spoken dialogue systems. Here’s a summary of her career and contributions from her early PhD years to the present:\n\n1. **Early Research and PhD**:\n   - Tiziana Ligorio's doctoral research focused on feature selection for spoken dialogue systems. This area involves the interaction between humans and machines, particularly how machines can understand and process spok

## Using MCP Server
  MCP (Model Context Protocol) is an open standard developed by Anthropic that allows AI models to connect to external tools and data sources through a unified, portable protocol. Unlike LangChain's built-in tool wrappers, MCP servers are standalone processes that any MCP-compatible client can use, not just LangChain.   
  So far, our agent has relied on LangChain's built-in wrappers for SerpAPI and Wikipedia, which return short search snippets.   
  For demonstration purposes, we use an MCP server yo **fetch the full content of a web page**, which LangChain has no built-in tool for.

In [17]:
%%capture
!uv pip install langchain-mcp-adapters mcp nest_asyncio


The MCP connection code below is **asynchronous**. This is not specific to the `mcp-server-fetch` server — it is a requirement of two things working together:

- The **MCP Python SDK** (`mcp.client.stdio`, `ClientSession`) is built on asyncio, because connecting to an MCP server means spawning a subprocess and communicating with it over stdin/stdout pipes. Reading from and writing to those pipes is I/O-bound work, and asyncio allows the program to wait for that I/O without blocking everything else.
- **`load_mcp_tools`** from `langchain_mcp_adapters` is also an async function — it needs to send a request to the server asking what tools are available, then wait for the response.

The same async pattern would apply regardless of which MCP server you connect to.   
In addition, we use `nest_asyncio` to allow `asyncio.run()` to work inside Jupyter, which already runs its own event loop.

In [ ]:
import asyncio
import nest_asyncio
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# Allow asyncio.run() to work inside Jupyter's existing event loop
nest_asyncio.apply()

# Connect to mcp-server-fetch: a standalone MCP server that retrieves the full content of a URL
# uvx downloads and runs the server automatically (no separate install needed)
# This is all you need to change to use a different mcp tool
server_params = StdioServerParameters(
    command="uvx",
    args=["mcp-server-fetch"],
)

# AsyncExitStack keeps the MCP connection open across cells
# so the fetch tool remains usable when the agent invokes it
_mcp_stack = AsyncExitStack()

""" await is a keyword used inside async functions to pause execution at that
  point and hand control back to the event loop until the async operation       
  completes, without blocking the rest of the program. """

async def connect_to_fetch_server():
    read, write = await _mcp_stack.enter_async_context(stdio_client(server_params))
    session = await _mcp_stack.enter_async_context(ClientSession(read, write))
    await session.initialize()
    return await load_mcp_tools(session)

fetch_tool = asyncio.run(connect_to_fetch_server())

for t in fetch_tool:
    t.handle_tool_error = True



### Side Note - Where to look for MCP servers

 The authoritative source is the official MCP servers repo:
  https://github.com/modelcontextprotocol/servers

  It lists all officially maintained servers with the exact command and args to
  use for each one. Each server's README shows the StdioServerParameters snippet
   (or equivalent) ready to copy.

| Command | Package registry | Example |                                      
  |---|---|---|                                                               
  | `uvx` | PyPI | `"mcp-server-fetch"` |                                       
  | `npx` | npm | `"@modelcontextprotocol/server-github"` |                     
  | direct path | local | `"/path/to/server.py"` |                              
                                                     

In [19]:
# before adding the MCP fetch tool
[t.name for t in tools]


['Search', 'wikipedia']

In [20]:
for ft in fetch_tool:
    tools.append(ft)


In [21]:
# after adding the MCP fetch tool
[t.name for t in tools]


['Search', 'wikipedia', 'fetch']

In [23]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a research assistant.\n"
        "For research questions, do not rely on a single source.\n"
        "Use multiple search or information-gathering tool calls to cover "
        "different time periods, perspectives, or subtopics before answering.\n"
        "When a search result returns a promising URL, use the fetch tool to "
        "retrieve the full page content for deeper information.\n"
        "If a fetch returns a 404 error, that URL does not exist — do NOT retry it. "
        "Move on to the next most promising URL from the search results, "
        "or proceed without fetching if no other URLs are available.\n"
        "Only provide a final answer after you have gathered information "
        "from multiple sources."
    )
)

In [ ]:
"""In a regular Python script, await can only appear inside an async def function; writing it at the top level 
  would raise a SyntaxError because the top-level scope is not an async context. Jupyter notebooks are different: 
  IPython runs each cell inside an event loop and wraps cell execution in an async context automatically, so
  top-level await in a cell is valid. 
  This is why answer = await agent.ainvoke(...) works directly in a cell, while the same line in a .py script would need to
  live inside an async def function"""

answer = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Who is Tiziana Ligorio? What has she been working on recently?"}
    ]
})

In [25]:
trace = build_agent_trace(answer)
trace


{'trace': [{'type': 'user',
   'content': 'Who is Tiziana Ligorio? What has she been working on recently?'},
  {'type': 'action',
   'tool': 'wikipedia',
   'args': {'query': 'Tiziana Ligorio'}},
  {'type': 'action',
   'tool': 'Search',
   'args': {'__arg1': 'Tiziana Ligorio recent work 2023'}},
  {'type': 'observation', 'tool': 'wikipedia', 'chars': 1521},
  {'type': 'observation', 'tool': 'Search', 'chars': 1448},
  {'type': 'action',
   'tool': 'Search',
   'args': {'__arg1': 'Tiziana Ligorio recent publications 2023'}},
  {'type': 'observation', 'tool': 'Search', 'chars': 1458},
  {'type': 'action',
   'tool': 'fetch',
   'args': {'url': 'https://www.hunter.cuny.edu/cs/faculty/tiziana-ligorio'}},
  {'type': 'action',
   'tool': 'fetch',
   'args': {'url': 'https://www.researchgate.net/profile/Tiziana-Ligorio'}},
  {'type': 'observation', 'tool': 'fetch', 'chars': 88},
  {'type': 'observation', 'tool': 'fetch', 'chars': 86},
  {'type': 'action',
   'tool': 'Search',
   'args': {'__

## Adding Custom Tool
For demonstration purposes, we add a custom tool to our search agent: a tool that scans search results for year mentions and groups the evidence by time period, helping the agent reason chronologically about the information it finds.  

We use the `@tool` decorator to register this function as a tool that the agent can call, allowing LangChain to expose its inputs and outputs in a structured way.
What required explicit JSON schemas and parsing logic in our previous demo is handled automatically by LangChain through a single @tool decorator.  
Note the importance of the docstring here, which effectively tells the Agent what this tool is and how/when to use it.

In [26]:
from typing import List, Dict, Any
from collections import defaultdict

@tool
def extract_year_timeline(results: List[str]) -> Dict[str, Any]:
    """
    Extract year mentions from search result snippets and group the snippets by year.

    Use this after a search tool call to quickly identify which time periods appear
    in the evidence (e.g., early years vs recent years) and to plan targeted follow-up searches.
    """
    year_to_snippets = defaultdict(list)
    year_pattern = re.compile(r"\b(19\d{2}|20\d{2})\b")

    for r in results:
        years = year_pattern.findall(r)
        for y in years:
            # keep a few distinct snippets per year
            if r not in year_to_snippets[y]:
                year_to_snippets[y].append(r)

    years_sorted = sorted(year_to_snippets.keys())
    return {
        "years": years_sorted,
        "by_year": dict(year_to_snippets),
        "count_years": len(years_sorted),
    }


### Add the custom tool to the Agent and re-run the query


In [27]:
# before adding the custom tool
[t.name for t in tools]


['Search', 'wikipedia', 'fetch']

In [28]:
tools.append(extract_year_timeline)

In [29]:
# after adding the custom tool
[t.name for t in tools]


['Search', 'wikipedia', 'fetch', 'extract_year_timeline']

Let's modify the system prompt to nudge the agent to use the custom tool as well.

In [30]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt = (
    "You are a research assistant.\n"
    "For research questions, do not rely on a single source.\n"
    "Use multiple search or information-gathering tool calls to cover "
    "different time periods, perspectives, or subtopics before answering.\n"
    "After a web search, if the results mention years or dates, use the "
    "extract_year_timeline tool to identify relevant time periods and guide "
    "any follow-up searches.\n"
    "When a search result returns a promising URL, use the fetch tool to "
    "retrieve the full page content for deeper information.\n"
    "If a fetch returns a 404 error, that URL does not exist — do NOT retry it. "
    "Move on to the next most promising URL from the search results, "
    "or proceed without fetching if no other URLs are available.\n"
    "Only provide a final answer after you have gathered information "
    "from multiple sources."
    )
)

In [31]:
inputs = {"messages": [{"role": "user", "content": "Who is Tiziana Ligorio? Focus on her computer science academic career and research, from her early PhD years until now. Discuss how her interests might have shifted based on what she has been teaching in last few years."}]}

answer = await agent.ainvoke(inputs)




In [32]:
trace_obj = build_agent_trace(answer)
trace_obj 

{'trace': [{'type': 'user',
   'content': 'Who is Tiziana Ligorio? Focus on her computer science academic career and research, from her early PhD years until now. Discuss how her interests might have shifted based on what she has been teaching in last few years.'},
  {'type': 'action',
   'tool': 'wikipedia',
   'args': {'query': 'Tiziana Ligorio'}},
  {'type': 'observation', 'tool': 'wikipedia', 'chars': 1521},
  {'type': 'action',
   'tool': 'Search',
   'args': {'__arg1': 'Tiziana Ligorio computer science academic career research'}},
  {'type': 'observation', 'tool': 'Search', 'chars': 1520},
  {'type': 'action',
   'tool': 'extract_year_timeline',
   'args': {'results': ['Tiziana Ligorio is a doctoral lecturer in the Department of Computer Science.',
     'Research Areas. Machine Learning; Spoken Dialogue Systems; Constraint ...',
     "Tiziana Ligorio's 13 research works with 50 citations, including: Naturalistic Dialogue Management for Noisy Speech Recognition.",
     'Teaching h

# --------------------------------------------------------------------
# Part 2: Observability and Evaluation
## *We will continue exploring the rest of this notebook after our lecture on evals.*

LangSmith is LangChain’s observability and evaluation platform for agent-based systems. It records everything that happens during an agent run (LLM calls, tool calls, intermediate states, timing, and token usage) so you can inspect how an agent behaved, not just what it returned. It can be used for debugging agents, comparing different prompts or tools, evaluating behavior across datasets, and understanding failure modes in complex, multi-step workflows. 

LangSmith works by attaching tracing callbacks to the runtime and capturing execution events as they occur. Because Google Colab replaces Python’s standard output system, this tracing mechanism is unstable in Colab notebooks and can cause runtime errors even when the agent itself is correct. For this reason, LangSmith tracing is not supported reliably in Colab. To use LangSmith, you should run the local version of this notebook from the course GitHub repository (for example using a local virtual environment and Jupyter or VS Code), where tracing works correctly and you can inspect full agent traces in the LangSmith UI.

# Structured Tracing with LangSmith 

In Part 1, we built a simple `build_agent_trace()` function to inspect agent behavior. 

In this section, we'll enable LangSmith tracing and re-run the agent to inspect with LangSmith.

## Get a LangSmith API Key

1. Go to https://smith.langchain.com

2. Sign in with your GitHub, Google, or email account (or create a new account)

3. Once logged in, click on your profile icon in the bottom left corner

4. Select **Settings**

5. Navigate to **API Keys**

6. Click **+ API Key**

7. Give the key a name, e.g. `langchain-react_agent_demo`

8. Copy the key immediately (you won't be able to see it again)

**Important: Treat this key like a password. Do not share it, paste it into notebooks, or commit it to GitHub.**

## Add the LangSmith API Key

### In Colab:

1. On the left sidebar, click 🔑 Secrets

2. Add a new secret:

*   Name: LANGSMITH_API_KEY
*   Value: your actual API key

3. Toggle the switch to the left to give notebook access (you should see a checkmark)

### If running locally — Add to .env

Add the following to your `.env` file (replace with your own key):

`LANGSMITH_API_KEY=your_langsmith_key_here`

**Important: Never paste API keys into code cells.**

## Enable LangSmith Tracing

Setting these environment variables tells LangChain to automatically send trace data to LangSmith. The `LANGCHAIN_PROJECT` groups your runs together for easier organization.

In [ ]:
# LangSmith tracing environment variables were configured at startup (at the top of this notebook),
# right after load_dotenv(), to avoid kernel I/O conflicts mid-session.


## Re-run the Agent with Tracing

Now when we invoke the agent, LangSmith will automatically capture the full execution trace. After running this cell, go to https://smith.langchain.com and select the `react-agent-demo` project to see the trace.

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt = (
    "You are a research assistant.\n"
    "For research questions, do not rely on a single source.\n"
    "Use multiple search or information-gathering tool calls to cover "
    "different time periods, perspectives, or subtopics before answering.\n"
    "Only provide a final answer after you have gathered information "
    "from multiple sources."
    )
)

In [ ]:
# Run the agent - this will be traced in LangSmith
answer = await agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Who is Tiziana Ligorio?"}
    ]
})

# Display the final answer
print(answer["messages"][-1].content)

Tiziana Ligorio is a multifaceted individual with a background in both the arts and computer science. Here are some key points about her:

1. **Academic Background**: Tiziana Ligorio is a Doctoral Lecturer in the Department of Computer Science at Hunter College, part of The City University of New York. She holds a PhD and has been involved in teaching courses related to computer science and machine learning.

2. **Research Contributions**: She has made significant contributions to the field of computer science, particularly in areas such as naturalistic dialogue management and has co-authored papers that have received recognition, including awards at conferences like UIST and ACM CHI.

3. **Artistic Involvement**: Tiziana Ligorio has also been involved in the art world, having attended events such as the opening of the "After Nature" exhibition at the New Museum in New York City in 2008.

4. **Language Proficiency**: She is proficient in multiple languages, including Italian (native), 

In [ ]:
import os
print(os.getenv("LANGSMITH_ENDPOINT"))


https://api.smith.langchain.com


## Simple Evaluation

Beyond tracing individual runs, LangSmith allows you to evaluate your agent systematically against a dataset. For demonstration purposes, we'll create a small test set and run a simple evaluator that checks whether the agent's response contains expected facts.

This pattern is useful for:
- Regression testing (i.e. re-running known test cases to ensure that previous correct behaviors have not degraded due to the change) after prompt or model changes
- Comparing different models or configurations
- Extensive testing before deploying agents to production

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

# evaluate() requires a LangSmith dataset — it cannot accept plain Python dicts.
# We create a named dataset via the API and upload our test cases to it.
dataset_name = "react-agent-test-set"

# Delete and recreate so re-running the notebook doesn't accumulate duplicate examples
if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)

dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    inputs=[
        {"input": "Who is Tiziana Ligorio?"},
        {"input": "What is reinforcement learning?"},
    ],
    outputs=[
        {"expected_facts": ["Hunter College", "computer science", "PhD"]},
        {"expected_facts": ["reward", "agent", "learning"]},
    ],
    dataset_id=dataset.id,
)


{'example_ids': ['15f2265d-7550-4184-8c8e-a84b62b98821',
  '9e232c9c-6789-4f2b-89f0-fe5952823595'],
 'count': 2}

In [ ]:
# Simple evaluator: checks what fraction of expected facts appear in the answer
def fact_coverage_evaluator(run, example):
    """
    Evaluates whether the agent's response contains the expected facts.
    Returns a score between 0 and 1 representing the fraction of facts found.
    """
    final_message = run.outputs["messages"][-1]
    answer = final_message.content.lower() if hasattr(final_message, "content") else str(final_message).lower()

    # expected_facts are stored in example.outputs (not example.inputs)
    expected = example.outputs["expected_facts"]
    found = sum(1 for fact in expected if fact.lower() in answer)

    return {
        "key": "fact_coverage",
        "score": found / len(expected),
        "comment": f"Found {found}/{len(expected)} expected facts",
    }

In [ ]:
# Wrapper so evaluate() can call our agent
def run_agent(inputs):
    return agent.invoke({
        "messages": [{"role": "user", "content": inputs["input"]}]
    })

### Run the Evaluation

The `evaluate` function will:
1. Run the agent on each example in our test set
2. Apply our custom evaluator to each result
3. Record everything in LangSmith for analysis

After running, visit LangSmith to see the evaluation results, including scores and traces for each test case.

In [ ]:
# Run the evaluation
results = evaluate(
    run_agent,
    data=dataset_name,
    evaluators=[fact_coverage_evaluator],
    experiment_prefix="react-agent-eval",
)

print("Evaluation complete! Check LangSmith for detailed results.")


View the evaluation results for experiment: 'react-agent-eval-50a4cda7' at:
https://smith.langchain.com/o/4faa9f56-b77c-482c-bd99-d16f4a864b7e/datasets/17a5b126-5fc7-4f9d-9339-9f2a535a45dd/compare?selectedSessions=15e21a44-2a6c-4e96-b872-252b9636a3ab




0it [00:00, ?it/s]

Evaluation complete! Check LangSmith for detailed results.


## Summary

**In Part 1**, we built a ReAct agent using the LangChain framework:              
   
  1. Framework setup: By calling create_agent with a model, a list of tools, and
   a short system prompt, we got a fully functional ReAct agent — the reasoning
  loop, tool routing, and termination logic are all handled by the framework 
  2. Built-in tools: We loaded LangChain's built-in wrappers for web search
  (SerpAPI) and Wikipedia with a single load_tools call, and connected an MCP
  server to add a fetch tool for retrieving full page content.
  3. Custom tools: We defined a custom extract_year_timeline tool using the
  @tool decorator. LangChain automatically exposes its inputs and outputs to
  the agent based on the type hints and docstring.
  4. Inspecting agent behavior: We built a build_agent_trace() helper to parse
  the agent's raw message output into a readable step-by-step trace of user
  input, actions, and observations.

  These techniques illustrate:
  - Quickly prototyping agents without implementing the reasoning loop from
  scratch (less coding but also less control)
  - Extending agents with tools: built-in, MCP-based, or fully custom
  - Inspecting agent decisions to debug unexpected behavior before adding full
  observability


**In Part 2**, we explored structured observability with LangSmith:

1. **Tracing**: By setting a few environment variables, we enabled automatic capture of all LLM calls, tool invocations, latencies, and token usage, without modifying our agent code.

2. **Evaluation**: We created a simple test dataset with expected facts and wrote a custom evaluator to measure how well our agent's responses covered those facts.

These techniques are useful for:
- Use tracing to extensively debug unexpected agent behavior
- Build regression test suites to catch issues before deployment
- Compare different models, prompts, or tool configurations systematically
